<a href="https://colab.research.google.com/github/yilinw762/ds2002-fa26/blob/main/notebooks/04-json-apis/2026_09_30_%E2%80%94_JSON_to_Tidy_Tables_%E2%80%94_Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · JSON to Tidy Tables

**Studio — 2026-09-30 · Fall 2026**  
**Class time:** 45 minutes

---

## From one API response to a table you can join

Today's payload is a single response of the kind you will get on the midterm and capstone: metadata at the top, two levels of nesting, some records missing fields, and one list that somebody stored as a string.

You are producing two tables from it — one row per vendor, and one row per line item — and then joining them. Everything is guarded by row-count checks, because nested data hides lost rows better than anything else you will work with.

In [1]:
import json, pandas as pd

response = {
    'game': 'UVA vs Duke',
    'date': '2026-10-10',
    'attendance': 47200,
    'zones': [
        {'zone': 'A', 'capacity': 400, 'vendors': [
            {'vendor_id': 'V-01', 'name': 'Hoos Burgers',
             'sales': {'orders': 300, 'revenue': 2250.0},
             'lines': [{'item': 'Cheeseburger', 'qty': 180},
                       {'item': 'Hot Dog', 'qty': 120}]},
            {'vendor_id': 'V-07', 'name': 'Stadium Dogs',
             'sales': {'orders': 210},
             'lines': [{'item': 'Hot Dog', 'qty': 210}]},
        ]},
        {'zone': 'B', 'capacity': 150, 'vendors': [
            {'vendor_id': 'V-12', 'name': 'Wahoo Wings',
             'sales': {'orders': 150, 'revenue': 1800.0},
             'lines': [{'item': 'Wings 6pc', 'qty': 90},
                       {'item': 'Wings 12pc', 'qty': 60}]},
        ]},
        {'zone': 'C', 'capacity': 120, 'vendors': [
            {'vendor_id': 'V-18', 'name': 'Rally Rain Gear',
             'sales': {'orders': 95, 'revenue': 570.0},
             'lines': [{'item': 'Rain Poncho', 'qty': 95}]},
        ]},
    ],
}
print(json.dumps(response, indent=2)[:400], '...')

{
  "game": "UVA vs Duke",
  "date": "2026-10-10",
  "attendance": 47200,
  "zones": [
    {
      "zone": "A",
      "capacity": 400,
      "vendors": [
        {
          "vendor_id": "V-01",
          "name": "Hoos Burgers",
          "sales": {
            "orders": 300,
            "revenue": 2250.0
          },
          "lines": [
            {
              "item": "Cheeseburger",
        ...


### Step 0 — map the shape before you touch it

Never normalize a payload you have not walked. These two cells tell you the nesting depth and how many rows to expect at each level, which is what your assertions will check against.

In [2]:
print('top-level keys:', list(response.keys()))
print('zones:', len(response['zones']))
print('vendors:', sum(len(z['vendors']) for z in response['zones']))
print('line items:', sum(len(v['lines'])
                         for z in response['zones'] for v in z['vendors']))

top-level keys: ['game', 'date', 'attendance', 'zones']
zones: 3
vendors: 4
line items: 6


Four vendors, six line items. Write those numbers down — every step below gets checked against them.

**TODO:** what is one row of each table you are about to build?

- Table 1: one row per _..._
- Table 2: one row per _..._

### Build 1 — one row per vendor

**TODO:** flatten to one row per vendor. Carry down the game, the date, and the zone's own `zone` and `capacity`. Then rename the meta columns to something readable.

*Expected: 4 rows.*

In [3]:
# TODO: vendors = pd.json_normalize(response, record_path=..., meta=[...])
# TODO: rename the zones.* columns
# TODO: assert len(vendors) == 4
vendors = pd.json_normalize(
    response,
    record_path=['zones', 'vendors'],
    meta=['game', 'date', ['zones', 'zone'], ['zones', 'capacity']]
)

vendors = vendors.rename(columns={
    'zones.zone': 'zone',
    'zones.capacity': 'capacity'
})

assert len(vendors) == 4
vendors


,vendor_id,name,lines,sales.orders,sales.revenue,game,date,zone,capacity
0,V-01,Hoos Burgers,"[{'item': 'Cheeseburger', 'qty': 180}, {'item'...",300,2250.0,UVA vs Duke,2026-10-10,A,400
1,V-07,Stadium Dogs,"[{'item': 'Hot Dog', 'qty': 210}]",210,NaN,UVA vs Duke,2026-10-10,A,400
2,V-12,Wahoo Wings,"[{'item': 'Wings 6pc', 'qty': 90}, {'item': 'W...",150,1800.0,UVA vs Duke,2026-10-10,B,150
3,V-18,Rally Rain Gear,"[{'item': 'Rain Poncho', 'qty': 95}]",95,570.0,UVA vs Duke,2026-10-10,C,120


### Build 2 — find what the API did not tell you

One vendor is missing a field. **TODO:** print the null count per column, identify which vendor and which field, and add a boolean column that records whether revenue was reported.

Do **not** fill it with zero. Explain in a comment why that would be wrong here.

In [4]:
# TODO
print(vendors.isnull().sum())

missing_revenue = vendors.loc[
    vendors['sales.revenue'].isna(),
    ['vendor_id', 'name', 'sales.revenue']
]
print('\nVendor with missing revenue:')
print(missing_revenue)

vendors['revenue_reported'] = vendors['sales.revenue'].notna()

vendors


vendor_id        0
name             0
lines            0
sales.orders     0
sales.revenue    1
game             0
date             0
zone             0
capacity         0
dtype: int64

Vendor with missing revenue:
  vendor_id          name  sales.revenue
1      V-07  Stadium Dogs            NaN


,vendor_id,name,lines,sales.orders,sales.revenue,game,date,zone,capacity,revenue_reported
0,V-01,Hoos Burgers,"[{'item': 'Cheeseburger', 'qty': 180}, {'item'...",300,2250.0,UVA vs Duke,2026-10-10,A,400,True
1,V-07,Stadium Dogs,"[{'item': 'Hot Dog', 'qty': 210}]",210,NaN,UVA vs Duke,2026-10-10,A,400,False
2,V-12,Wahoo Wings,"[{'item': 'Wings 6pc', 'qty': 90}, {'item': 'W...",150,1800.0,UVA vs Duke,2026-10-10,B,150,True
3,V-18,Rally Rain Gear,"[{'item': 'Rain Poncho', 'qty': 95}]",95,570.0,UVA vs Duke,2026-10-10,C,120,True


### Build 3 — one row per line item

**TODO:** flatten the same payload again, this time down to line items, keeping `vendor_id` and `zone` on every row.

*Expected: 6 rows, and the quantities should sum to 755.*

In [5]:
# TODO
# TODO: assert len(lines) == 6
# TODO: assert lines['qty'].sum() == 755
lines = pd.json_normalize(
    response,
    record_path=['zones', 'vendors', 'lines'],
    meta=[['zones', 'vendors', 'vendor_id'], ['zones', 'zone']]
)

lines = lines.rename(columns={
    'zones.vendors.vendor_id': 'vendor_id',
    'zones.zone': 'zone'
})

assert len(lines) == 6
assert lines['qty'].sum() == 755
lines


,item,qty,vendor_id,zone
0,Cheeseburger,180,V-01,A
1,Hot Dog,120,V-01,A
2,Hot Dog,210,V-07,A
3,Wings 6pc,90,V-12,B
4,Wings 12pc,60,V-12,B
5,Rain Poncho,95,V-18,C


### Build 4 — join the two tables

**TODO:** attach the vendor name and the zone capacity to each line item. Use `indicator=True` and confirm you still have 6 rows and nothing failed to match.

In [6]:
# TODO
joined = lines.merge(
    vendors[['vendor_id', 'name', 'capacity']],
    on='vendor_id',
    how='left',
    indicator=True
)

assert len(joined) == 6
assert (joined['_merge'] == 'both').all()

print(joined['_merge'].value_counts())
joined


_merge
both          6
left_only     0
right_only    0
Name: count, dtype: int64


,item,qty,vendor_id,zone,name,capacity,_merge
0,Cheeseburger,180,V-01,A,Hoos Burgers,400,both
1,Hot Dog,120,V-01,A,Hoos Burgers,400,both
2,Hot Dog,210,V-07,A,Stadium Dogs,400,both
3,Wings 6pc,90,V-12,B,Wahoo Wings,150,both
4,Wings 12pc,60,V-12,B,Wahoo Wings,150,both
5,Rain Poncho,95,V-18,C,Rally Rain Gear,120,both


### Build 5 — the list that came in as a string

A second source exported item lists as text, because someone wrote them to a CSV. **TODO:** turn this into one row per item — split the string, then `explode`.

In [7]:
csv_ish = pd.DataFrame({
    'order_id': [501, 502],
    'items': ['Cheeseburger|Soda|Fries', 'Rain Poncho'],
})
csv_ish

,order_id,items
0,501,Cheeseburger|Soda|Fries
1,502,Rain Poncho


In [8]:
# TODO: split on '|', explode, and clean up the index
# TODO: assert the result has 4 rows
items = (
    csv_ish
    .assign(items=csv_ish['items'].str.split('|'))
    .explode('items')
    .reset_index(drop=True)
)

assert len(items) == 4
items


,order_id,items
0,501,Cheeseburger
1,501,Soda
2,501,Fries
3,502,Rain Poncho


### Build 6 — answer something

**TODO:** using your joined line-item table, print units sold per zone and the single best-selling item overall. Then say in one sentence what zone C's numbers suggest about the weather that day.

In [9]:
# TODO
units_by_zone = joined.groupby('zone')['qty'].sum()
print('Units sold per zone:')
print(units_by_zone)

best_selling_item = (
    joined.groupby('item', as_index=False)['qty']
    .sum()
    .sort_values('qty', ascending=False)
    .iloc[0]
)
print('\nBest-selling item overall:')
print(f"{best_selling_item['item']}: {best_selling_item['qty']} units")

print("\nZone C sold 95 Rain Ponchos, which suggests it was likely rainy or wet that day.")


Units sold per zone:
zone
A    510
B    150
C     95
Name: qty, dtype: int64

Best-selling item overall:
Hot Dog: 330 units

Zone C sold 95 Rain Ponchos, which suggests it was likely rainy or wet that day.


---

## Checkpoint (participation)

Report your two row counts and what you did about the vendor with no revenue reported.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [10]:
# Checkpoint
vendor_rows = len(vendors)
line_item_rows = len(lines)
units_total = lines['qty'].sum()
missing_revenue_call = (
    'Left Stadium Dogs revenue as NaN and added revenue_reported=False '
    'because missing revenue does not mean $0 revenue.'
)

print('vendor rows:', vendor_rows)
print('line item rows:', line_item_rows)
print('total units:', units_total)
print('missing revenue:', missing_revenue_call)


vendor rows: 4
line item rows: 6
total units: 755
missing revenue: Left Stadium Dogs revenue as NaN and added revenue_reported=False because missing revenue does not mean $0 revenue.
